[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014/blob/main/notebooks/week5_transportation.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "paste your ACCESSID here",
    "WLSSECRET":   "paste your SECRET here",
    "LICENSEID":   123456,   # the number itself, no quotes
}
env = gp.Env(params=params)

# Transportation — the game's board

Two depots ship crates to three divisions. Every depot can reach every division, and crates ship whole $\Rightarrow x_{ij} \in \mathbb{Z}_+$.

|  | Div 1 | Div 2 | Div 3 | supply |
| :--- | ---: | ---: | ---: | ---: |
| Depot A | 4 | 5 | 9 | **5** |
| Depot B | 3 | 6 | 6 | **7** |
| demand | **6** | **2** | **4** | 12 = 12 |

Inner cells = cost per crate $c_{ij}$. $\;x_{A1}$ = crates shipped from depot A to division 1 (the note's $x_{11}$).

$$
\begin{aligned}
\min_{x} \quad & 4x_{A1} + 5x_{A2} + 9x_{A3} + 3x_{B1} + 6x_{B2} + 6x_{B3} \\
\text{s.t.} \quad & x_{A1} + x_{A2} + x_{A3} \le 5 \qquad \text{depot A ships no more than it holds} \\
                  & x_{B1} + x_{B2} + x_{B3} \le 7 \qquad \text{depot B} \\
                  & x_{A1} + x_{B1} \ge 6 \qquad \text{division 1 gets what it needs} \\
                  & x_{A2} + x_{B2} \ge 2 \qquad \text{division 2} \\
                  & x_{A3} + x_{B3} \ge 4 \qquad \text{division 3} \\
                  & x_{A1},\, x_{A2},\, x_{A3},\, x_{B1},\, x_{B2},\, x_{B3} \in \mathbb{Z}_+
\end{aligned}
$$

**Math to code**

| math | code |
| :--- | :--- |
| $x_{ij} \in \mathbb{Z}_+$ | `addVar(vtype=GRB.INTEGER)` — lower bound is already `0` |
| $\min$ total cost | `setObjective(expr, GRB.MINIMIZE)` — expression **and** sense |
| each $\le$ / $\ge$ row | one `addConstr` — same coefficients, same order |

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# decision variables, six integer variables x_A1, ..., x_B3 --- e.g., ip.addVar(vtype=GRB.INTEGER)


# let's define objective function, total shipping cost --- ip.setObjective(..., GRB.MINIMIZE)


# supply constraints, one per depot --- ip.addConstr(... <= ...)


# demand constraints, one per division --- ip.addConstr(... >= ...)


# Model is set up. Let's optimize --- ip.optimize()


# Let's print the six shipments and the total cost

## Reference: the completed model

By hand — no dictionaries, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
ip = gp.Model("transportation_ip")

# decision variables -- crates on each road, whole numbers, lower bound 0 by default
x_A1 = ip.addVar(vtype=GRB.INTEGER, name="x_A1")
x_A2 = ip.addVar(vtype=GRB.INTEGER, name="x_A2")
x_A3 = ip.addVar(vtype=GRB.INTEGER, name="x_A3")
x_B1 = ip.addVar(vtype=GRB.INTEGER, name="x_B1")
x_B2 = ip.addVar(vtype=GRB.INTEGER, name="x_B2")
x_B3 = ip.addVar(vtype=GRB.INTEGER, name="x_B3")

# objective -- total shipping cost, written out term by term
ip.setObjective(4 * x_A1 + 5 * x_A2 + 9 * x_A3 + 3 * x_B1 + 6 * x_B2 + 6 * x_B3, GRB.MINIMIZE)

# supply -- each depot ships no more than it holds
ip.addConstr(x_A1 + x_A2 + x_A3 <= 5, name="supply_A")
ip.addConstr(x_B1 + x_B2 + x_B3 <= 7, name="supply_B")

# demand -- each division gets what it needs
ip.addConstr(x_A1 + x_B1 >= 6, name="demand_1")
ip.addConstr(x_A2 + x_B2 >= 2, name="demand_2")
ip.addConstr(x_A3 + x_B3 >= 4, name="demand_3")

ip.optimize()

print(f"\nA -> 1: {round(x_A1.X)}   A -> 2: {round(x_A2.X)}   A -> 3: {round(x_A3.X)}")
print(f"B -> 1: {round(x_B1.X)}   B -> 2: {round(x_B2.X)}   B -> 3: {round(x_B3.X)}")
print(f"total cost = {ip.ObjVal:.0f}")